# Data Understanding / Initial EDA

In [33]:
import kagglehub
import pandas as pd
import seaborn as sns
from pathlib import Path
from datetime import datetime
import plotly.graph_objects as go

In [2]:
# Load datasets
path = kagglehub.dataset_download("anikannal/cms-synthetic-data")

# Read in datasets
datasets = {
    fp.stem: pd.read_csv(fp)
    for fp in Path(path).glob("*.csv")
}

C:\Users\steph\AppData\Local\Temp\ipykernel_8936\3649949676.py:6: DtypeWarning: Columns (21: ICD9_DGNS_CD_10, 23: ICD9_PRCDR_CD_2, 24: ICD9_PRCDR_CD_3, 25: ICD9_PRCDR_CD_4, 26: ICD9_PRCDR_CD_5, 27: ICD9_PRCDR_CD_6) have mixed types. Specify dtype option on import or set low_memory=False.
  fp.stem: pd.read_csv(fp)


In [3]:
# Extract general information
def overview(datasets: dict):
    """Extract general overview information from a collection of datasets"""
    overview = {
        k:[]
        for k in ["dataset", "shape", "unique_rows"]
    }
    for name,df in datasets.items():
        overview["dataset"].append(name)
        overview["shape"].append(df.shape)
        overview["unique_rows"].append(len(df.drop_duplicates()))
    return pd.DataFrame(overview)

dataset_overview = overview(datasets)
dataset_overview

,dataset,shape,unique_rows
0,DE1_0_2008_Beneficiary_Summary_File_Sample_1,"(116352, 32)",116352
1,DE1_0_2008_to_2010_Inpatient_Claims_Sample_1,"(66773, 81)",66773
2,DE1_0_2008_to_2010_Outpatient_Claims_Sample_1,"(790790, 76)",790790
3,DE1_0_2009_Beneficiary_Summary_File_Sample_1,"(114538, 32)",114538
4,DE1_0_2010_Beneficiary_Summary_File_Sample_1,"(112754, 32)",112754


In [4]:
# Combine similar datasets
datasets["beneficiary_summary"] = pd.concat([d for n,d in datasets.items() if "Beneficiary_Summary_File" in n])
print(f"New shape: {datasets["beneficiary_summary"].shape}")

# Filter down datasets
datasets = {k:v for k,v in datasets.items() if "Beneficiary_Summary_File" not in k}
dataset_overview = overview(datasets)
dataset_overview

New shape: (343644, 32)


,dataset,shape,unique_rows
0,DE1_0_2008_to_2010_Inpatient_Claims_Sample_1,"(66773, 81)",66773
1,DE1_0_2008_to_2010_Outpatient_Claims_Sample_1,"(790790, 76)",790790
2,beneficiary_summary,"(343644, 32)",322128


In [15]:
# Attributes
attributes = {}
for name,df in datasets.items():
    attr_data = {
        k:[]
        for k in ["attribute", "dtype", "empty_vals"]
    }
    for col in df.columns:
        attr_data["attribute"].append(col)
        attr_data["dtype"].append(df[col].dtype)
        attr_data["empty_vals"].append(sum(df[col].isna()))
    attr_df = pd.DataFrame(attr_data)
    attr_df = pd.merge(attr_df, df.describe().T.rename_axis("attribute").reset_index(), on="attribute")
    attributes[name] = attr_df

    print(name)
    display(attr_df)
    print("")

DE1_0_2008_to_2010_Inpatient_Claims_Sample_1


,attribute,dtype,empty_vals,count,mean,std,min,25%,50%,75%,max
0,CLM_ID,int64,0,66773.0,1.965017e+14,2.859526e+11,1.960112e+14,1.962512e+14,1.965012e+14,1.967512e+14,1.969912e+14
1,SEGMENT,int64,0,66773.0,1.001018e+00,3.189598e-02,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00
2,CLM_FROM_DT,float64,68,66705.0,2.008846e+07,7.587457e+03,2.007113e+07,2.008081e+07,2.009032e+07,2.009111e+07,2.010123e+07
3,CLM_THRU_DT,float64,68,66705.0,2.008861e+07,7.559295e+03,2.008010e+07,2.008082e+07,2.009032e+07,2.009112e+07,2.010123e+07
4,CLM_PMT_AMT,float64,0,66773.0,9.573633e+03,9.315073e+03,-8.000000e+03,4.000000e+03,7.000000e+03,1.100000e+04,5.700000e+04
...,...,...,...,...,...,...,...,...,...,...,...
57,HCPCS_CD_41,float64,66773,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58,HCPCS_CD_42,float64,66773,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,HCPCS_CD_43,float64,66773,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
60,HCPCS_CD_44,float64,66773,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



DE1_0_2008_to_2010_Outpatient_Claims_Sample_1


,attribute,dtype,empty_vals,count,mean,std,min,25%,50%,75%,max
0,CLM_ID,int64,0,790790.0,5.425026e+14,2.858482e+11,5.420123e+14,5.422523e+14,5.425023e+14,5.427523e+14,5.429923e+14
1,SEGMENT,int64,0,790790.0,1.014230e+00,1.184382e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00
2,CLM_FROM_DT,float64,11253,779537.0,2.008925e+07,7.475135e+03,2.007121e+07,2.008092e+07,2.009043e+07,2.009120e+07,2.010123e+07
3,CLM_THRU_DT,float64,11253,779537.0,2.008929e+07,7.471623e+03,2.008010e+07,2.008093e+07,2.009043e+07,2.009121e+07,2.010123e+07
4,CLM_PMT_AMT,float64,0,790790.0,2.839246e+02,5.713928e+02,-1.000000e+02,4.000000e+01,8.000000e+01,2.000000e+02,3.300000e+03
5,NCH_PRMRY_PYR_CLM_PD_AMT,float64,0,790790.0,1.023976e+01,2.346684e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.400000e+04
6,AT_PHYSN_NPI,float64,17790,773000.0,4.975733e+09,2.874373e+09,1.024080e+05,2.521572e+09,4.904972e+09,7.501324e+09,9.999886e+09
7,OP_PHYSN_NPI,float64,656357,134433.0,4.947815e+09,2.890483e+09,3.258650e+05,2.474254e+09,4.870758e+09,7.485573e+09,9.999615e+09
8,OT_PHYSN_NPI,float64,533124,257666.0,4.904993e+09,2.889353e+09,1.024080e+05,2.443058e+09,4.774818e+09,7.491945e+09,9.999470e+09
9,NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,float64,0,790790.0,1.289849e-02,2.315506e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,8.000000e+02



beneficiary_summary


,attribute,dtype,empty_vals,count,mean,std,min,25%,50%,75%,max
0,BENE_BIRTH_DT,int64,0,343644.0,1.936435e+07,125333.931002,19090101.0,19281101.0,19360501.0,19420301.0,19831201.0
1,BENE_DEATH_DT,float64,338183,5461.0,2.009074e+07,8214.953993,20080101.0,20081001.0,20090701.0,20100401.0,20101201.0
2,BENE_SEX_IDENT_CD,int64,0,343644.0,1.553069e+00,0.497176,1.0,1.0,2.0,2.0,2.0
3,BENE_RACE_CD,int64,0,343644.0,1.285065e+00,0.755564,1.0,1.0,1.0,1.0,5.0
4,SP_STATE_CODE,int64,0,343644.0,2.569672e+01,15.584441,1.0,10.0,25.0,39.0,54.0
5,BENE_COUNTY_CD,int64,0,343644.0,3.664712e+02,266.006973,0.0,141.0,330.0,560.0,999.0
6,BENE_HI_CVRAGE_TOT_MONS,int64,0,343644.0,1.118733e+01,2.886072,0.0,12.0,12.0,12.0,12.0
7,BENE_SMI_CVRAGE_TOT_MONS,int64,0,343644.0,1.084085e+01,3.411733,0.0,12.0,12.0,12.0,12.0
8,BENE_HMO_CVRAGE_TOT_MONS,int64,0,343644.0,3.159421e+00,5.189713,0.0,0.0,0.0,10.0,12.0
9,PLAN_CVRG_MOS_NUM,int64,0,343644.0,8.527427e+00,5.192147,0.0,2.0,12.0,12.0,12.0


In [17]:
# All unique attributes
all_attrs = set(sum([list(d["attribute"]) for d in attributes.values()], []))
all_attrs

{'AT_PHYSN_NPI',
 'BENE_BIRTH_DT',
 'BENE_COUNTY_CD',
 'BENE_DEATH_DT',
 'BENE_HI_CVRAGE_TOT_MONS',
 'BENE_HMO_CVRAGE_TOT_MONS',
 'BENE_RACE_CD',
 'BENE_SEX_IDENT_CD',
 'BENE_SMI_CVRAGE_TOT_MONS',
 'BENRES_CAR',
 'BENRES_IP',
 'BENRES_OP',
 'CLM_ADMSN_DT',
 'CLM_FROM_DT',
 'CLM_ID',
 'CLM_PASS_THRU_PER_DIEM_AMT',
 'CLM_PMT_AMT',
 'CLM_THRU_DT',
 'CLM_UTLZTN_DAY_CNT',
 'HCPCS_CD_1',
 'HCPCS_CD_10',
 'HCPCS_CD_11',
 'HCPCS_CD_12',
 'HCPCS_CD_13',
 'HCPCS_CD_14',
 'HCPCS_CD_15',
 'HCPCS_CD_16',
 'HCPCS_CD_17',
 'HCPCS_CD_18',
 'HCPCS_CD_19',
 'HCPCS_CD_2',
 'HCPCS_CD_20',
 'HCPCS_CD_21',
 'HCPCS_CD_22',
 'HCPCS_CD_23',
 'HCPCS_CD_24',
 'HCPCS_CD_25',
 'HCPCS_CD_26',
 'HCPCS_CD_27',
 'HCPCS_CD_28',
 'HCPCS_CD_29',
 'HCPCS_CD_3',
 'HCPCS_CD_30',
 'HCPCS_CD_31',
 'HCPCS_CD_32',
 'HCPCS_CD_33',
 'HCPCS_CD_34',
 'HCPCS_CD_35',
 'HCPCS_CD_36',
 'HCPCS_CD_37',
 'HCPCS_CD_38',
 'HCPCS_CD_39',
 'HCPCS_CD_4',
 'HCPCS_CD_40',
 'HCPCS_CD_41',
 'HCPCS_CD_42',
 'HCPCS_CD_43',
 'HCPCS_CD_44',
 'HCPCS_CD

In [ ]:
# Distribution of attributes
for name,df in datasets.items():
    for col in df.columns:
        if df[col].dtype in ["int64", "float64"]:
            fig = go.Figure(
                data = [
                    go.Histogram(
                        x = df[col],
                    )
                ],
                layout = {
                    "title": f"{name}: {col}",
                    "xaxis_title": col,
                }
            )
            Path(f"plots/{name}").mkdir(parents=True, exist_ok=True)
            fig.write_html(f"plots/{name}/{col}.html")

In [41]:
def parse_dt(d, in_days=False):
    """Parse date"""
    t = datetime.strptime(str(d), "%Y%m%d").timestamp()
    if in_days:
        return t/86400
    return t

In [58]:
# Inpatient dataset 
inpatient_key = "DE1_0_2008_to_2010_Inpatient_Claims_Sample_1"
inpatient_df = datasets[inpatient_key]
inpatient_df

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45,hospital_stay
0,00013D2EFD8E45D1,196661176988405,1,20100312.0,20100313.0,2600GD,4000.0,0.0,3.139084e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,00016F745862898F,196201177000368,1,20090412.0,20090418.0,3900MB,26000.0,0.0,6.476809e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0
2,00016F745862898F,196661177015632,1,20090831.0,20090902.0,3900HM,5000.0,0.0,6.119985e+08,6.119985e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,00016F745862898F,196091176981058,1,20090917.0,20090920.0,3913XU,5000.0,0.0,4.971603e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
4,00016F745862898F,196261176983265,1,20100626.0,20100701.0,3900MB,16000.0,0.0,6.408400e+09,1.960860e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66768,FFF8B49458F9FFA7,196641177004609,1,20080329.0,20080331.0,0100UV,3000.0,0.0,2.509032e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
66769,FFF8B49458F9FFA7,196851176989979,1,20081012.0,20081013.0,0100UV,10000.0,0.0,3.401706e+08,6.624958e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
66770,FFFA950301FCA748,196341176990073,1,20100109.0,20100112.0,0300QJ,7000.0,0.0,9.059627e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
66771,FFFC880E13CA6278,196191176959044,1,20081012.0,20081020.0,36T2DK,10000.0,0.0,9.809003e+08,9.809003e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0


In [53]:
# Times each patient appears in dataset
stays_per_patient = inpatient_df["DESYNPUF_ID"].value_counts()
stays_per_patient

DESYNPUF_ID
CD57573A5B77AAFE    14
FC02104A9F15D57C    13
3A9D94388C426924    12
7D0B2F944DE86994    12
D7279B2238D4AF28    12
                    ..
FFF2813198D935D3     1
FFF46A3796CB82BB     1
FFFA950301FCA748     1
FFFC880E13CA6278     1
FFFE3ED9582AD46E     1
Name: count, Length: 37780, dtype: int64

In [56]:
# Assume readmission as num times a patient appears in dataset
multistay_patients = list(stays_per_patient[stays_per_patient > 1].index)

print(f"Number of readmitted patients: {len(multistay_patients)}")
print(f"Total number of patients: {len(stays_per_patient)}")
print(f"Readmitted rate: {len(multistay_patients) / len(stays_per_patient) * 100:.2f}%")

Number of readmitted patients: 15168
Total number of patients: 37780
Readmitted rate: 40.15%


In [61]:
# Calculate length of stay
inpatient_df["hospital_stay"] = inpatient_df["NCH_BENE_DSCHRG_DT"].apply(parse_dt, in_days=True) - inpatient_df["CLM_ADMSN_DT"].apply(parse_dt, in_days=True)
inpatient_df

,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45,hospital_stay
0,00013D2EFD8E45D1,196661176988405,1,20100312.0,20100313.0,2600GD,4000.0,0.0,3.139084e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,00016F745862898F,196201177000368,1,20090412.0,20090418.0,3900MB,26000.0,0.0,6.476809e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0
2,00016F745862898F,196661177015632,1,20090831.0,20090902.0,3900HM,5000.0,0.0,6.119985e+08,6.119985e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,00016F745862898F,196091176981058,1,20090917.0,20090920.0,3913XU,5000.0,0.0,4.971603e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
4,00016F745862898F,196261176983265,1,20100626.0,20100701.0,3900MB,16000.0,0.0,6.408400e+09,1.960860e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66768,FFF8B49458F9FFA7,196641177004609,1,20080329.0,20080331.0,0100UV,3000.0,0.0,2.509032e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
66769,FFF8B49458F9FFA7,196851176989979,1,20081012.0,20081013.0,0100UV,10000.0,0.0,3.401706e+08,6.624958e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
66770,FFFA950301FCA748,196341176990073,1,20100109.0,20100112.0,0300QJ,7000.0,0.0,9.059627e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
66771,FFFC880E13CA6278,196191176959044,1,20081012.0,20081020.0,36T2DK,10000.0,0.0,9.809003e+08,9.809003e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0
